# Discrimination pipeline

Run the built-in **discrimination** paradigm. Real StimPy sessions are resolved via `config.paths`; **if none are found (or a session fails to parse) we print the error and fall back to a simulated session** so every analysis cell still runs.

**For real data:** edit `~/.piepy/config.json` so `paths.presentation` / `paths.analysis` point at your dirs. (Real sessions need their full artifacts — e.g. opto-pattern images for opto sessions — or parsing will raise.)

In [ ]:
import os
import polars as pl

from piepy.core.config import config
from piepy.core.registry import get_session_class
from piepy.core.hub import Hub

from piepy.viz.plots import psychometric
from piepy.viz.trial.wheel_detection import trial_snapshot

pres = config.paths["presentation"][0]
print('presentation dir:', pres)

## Parse a single session
Builds the Session, parses each run, and stacks them onto one session clock. Wrapped defensively so a real-data hiccup prints a clear error instead of aborting.

In [ ]:
name = "250404_VB101_discrim_opto120_V1__no_cam_VO"
sess = get_session_class("discrimination")(name, load_flag=False)
df = sess.concatenate_runs("discrimination")
print(name, "->", df.shape)

## Plotting from a single Run/Session

### Plotting from a dataframe

In [ ]:
pr = psychometric(data=df,
                  x="diff_width",
                  outcome="outcome",
                  compare="opto",
                  success="correct",
                  fit_curve=True,
                  model="logistic",
                  palette=("#DD7703","#0164E5"),
                  label='Opto ')

#### You can pass ```kwargs``` that override the visual properties of the plots

This requires a bit of knowledge of the composition of the plot. Psychometric plot is made from an ```errorbar``` and ```line``` plot. You can target the style of these components by prefixing your ```kwargs``` with their names, e.g. ```errorbar_linewidth```, ```line_linestyle```.

In [ ]:
pr = psychometric(data=df,
                  x="diff_width",
                  outcome="outcome",
                  compare="opto",
                  success="correct",
                  fit_curve=True,
                  model="logistic",
                  palette=("#DD7703","#0164E5"),
                  errorbar_linewidth=9,
                  line_linestyle=":",
                  label='Opto ')

#### The returned ```PlotResult``` object has the data, statistical tests(if applicable) and the (fig,ax) tuple

In [ ]:
pr

### Plotting with accessor

You can pass a ```filterer``` argument to filter the data

In [ ]:
pr = sess.viz.reaction_time_cloud(filterer={"outcome":"correct"},
                                  x="diff_width",
                                  value="response_time",
                                  compare="opto",
                                  palette=("#090909","#4C4CC7"),
                                  bin_width=50,
                                  label='Opto',
                                  dodge_width=0.2,
                                  violin_widths=0.2,
                                  violin_showextrema=False,
                                  violin_showmedians=True,
                                  width=0.05,
                                  scatter_s=50,
                                  scatter_linewidth=0.3,
                                  scatter_edgecolor="#FFFFFF")

In [ ]:
df.columns

In [ ]:
pr = sess.viz.reaction_time_dist(filterer={"outcome":"correct",
                                           "diff_width":-10},
                                  value="response_time",
                                  comparing="opto",
                                  palette=("#090909","#4C4CC7"),
                                  bin_width=5,
                                  label='Opto ',
                                  alpha=0.8)

## Cohort across many sessions (`Hub`)
`Hub` runs each session in parallel and stacks them into one cohort table. It already isolates per-session failures (a bad session warns and is skipped).

In [ ]:
cohort = None
if candidates:
    try:
        hub = Hub("discrimination")
        hub.initialize([os.path.basename(c) for c in candidates], load_sessions=False)
        cohort = hub.data
        print('cohort:', None if cohort is None else cohort.shape)
    except Exception as e:
        print("Hub gather failed:", type(e).__name__, e)